In [1]:
import ast

import pandas as pd


In [2]:

# Load the CSV file
df = pd.read_csv("/mnt/D/Data_Science/Common/Data_exploring_RAG/data.csv")


In [3]:

# Function to safely parse dictionary string
def parse_question(val):
    if pd.isna(val):
        return pd.Series([None, None])
    try:
        # Convert string representation of dict to actual dict
        data = ast.literal_eval(str(val))
        return pd.Series([data.get("hindi"), data.get("english")])
    except (ValueError, SyntaxError):
        # Fallback if there are unescaped backslashes or malformed syntax
        return pd.Series([None, None])



In [4]:

# Create the new columns
df[["hindi", "english"]] = df["question"].apply(parse_question)

# Optional: drop the original 'question' column if no longer needed
# df.drop(columns=['question'], inplace=True)

# Preview the results
print(df[["question", "hindi", "english"]].head())

                                            question  \
0  {"hindi": "माना बिंदु $(4,1,0)$ से होकर जाने व...   
1  {"hindi": "यदि फलन $f(x)=\\log _{7}\\left(1-\\...   
2  {"hindi": "माना एक फलन $f$ के लिए $f(x)+3 f\\l...   
3  {"hindi": "माना $f(x)= \\begin{cases}(1+a x)^{...   
4  {"hindi": "माना एक अवकलनीय फलन $g$ के लिए $\\i...   

                                               hindi  \
0  माना बिंदु $(4,1,0)$ से होकर जाने वाली एक रेखा...   
1  यदि फलन $f(x)=\log _{7}\left(1-\log _{4}\left(...   
2  माना एक फलन $f$ के लिए $f(x)+3 f\left(\frac{24...   
3  माना $f(x)= \begin{cases}(1+a x)^{1 / x}, & x<...   
4  माना एक अवकलनीय फलन $g$ के लिए $\int_{0}^{x} g...   

                                             english  
0  Let a line passing through the point $(4,1,0)$...  
1  If the domain of the function $f(x)=\log _{7}\...  
2  Let $f$ be a function such that $f(x)+3 f\left...  
3  Let $f(x)=\left\{\begin{array}{l}(1+a x)^{1 / ...  
4  Let $g$ be a differentiable function such that..

In [5]:
# 2. Flag rows containing images
# Matches markdown syntax '![' or common image extensions (.jpg, .jpeg, .png, .webp)
image_pattern = r"!\[|\.(?:jpg|jpeg|png|webp)"

df["has_image"] = (
    df["question"].str.contains(image_pattern, regex=True, na=False)
    | df["hindi"].str.contains(image_pattern, regex=True, na=False)
    | df["english"].str.contains(image_pattern, regex=True, na=False)
    | df["options"].str.contains(image_pattern, regex=True, na=False)
)

In [6]:
# Check results
print(df["has_image"].value_counts())
print(df[df["has_image"]][["hindi", "english", "has_image"]].head())

has_image
False    2937
True      284
Name: count, dtype: int64
                                                hindi  \
27  द्रव्यमानों $\mathrm{m}_{1}=10 \mathrm{~kg}$ औ...   
28  एक कण का प्रक्षेपण कोण, ऊर्ध्वाधर अक्ष से $\ph...   
32  एक खुरदरे (रूक्ष) क्षैतिज तल पर रखे, द्रव्यमान...   
49  त्रिज्याओं $\mathrm{R}_{1}=6 \pi \mathrm{~m}$ ...   
53  $\mathrm{CH}_{3} \mathrm{COOH}$ एवं HCl (जलीय ...   

                                              english  has_image  
27  \n\n![](https://mydream-dev.s3.ap-south-1.amaz...       True  
28  The angle of projection of a particle is measu...       True  
32  A force of 49 N acts tangentially at the highe...       True  
49  A loop ABCDA, carrying current $\mathrm{I}=12 ...       True  
53  40 mL of a mixture of $\mathrm{CH}_{3} \mathrm...       True  


In [7]:
df.to_csv("/mnt/D/Data_Science/Common/Data_exploring_RAG/data_with_translations.csv", index=False, encoding="utf-8-sig")